In [1]:
from pathlib import Path 
import sys

sys.path.append(str(Path.cwd().parent))

from services.job_extraction_service import scrape_raw_job_content, extract_jobs_info
from services.job_search_service import search_jobs_by_queries, get_job_search_queries, get_canonical_job_url
from services.job_ingestion_service import filter_unseen_jobs
from configs.app_container import Container
from models.websearch_request import WebSearchParams
from helpers.utils import get_datetime_utc, get_formatted_datetime
from datetime import timedelta

e:\MS in CS\Semester-3\Period 1\LLM\group_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\MS in CS\Semester-3\Period 1\LLM\group_project\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Fetching 8 files: 100%|██████████| 8/8 [00:00<00:00, 1331.68it/s]
Default prompt name is set to 'document'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


Table 'jobs' already exists - skipping creation.
Table 'search_schedules' ready.


In [2]:
# queries = build_search_queries(["Master Thesis LLM", "Master Thesis AI"])
queries = get_job_search_queries()

In [3]:
queries

['se.linkedin.com/jobs/ "Master Thesis AI" sweden']

In [4]:
end = get_formatted_datetime(get_datetime_utc())
start = get_formatted_datetime(get_datetime_utc() - timedelta(2))

start, end

('2026-09-26', '2026-09-28')

In [5]:
request = WebSearchParams(
    queries=queries,
    start_date=start,
    end_date=end
)

In [6]:
res = search_jobs_by_queries(request, print_result=True)


*********************** 1 **************************
Title: Master Thesis: Systematizing Neuro-Symbolic AI for Autonomous Network Management på Ericsson – Stockholm | LinkedIn Jobb
URL: https://www.linkedin.com/jobs/view/master-thesis-systematizing-neuro-symbolic-ai-for-autonomous-network-management-at-ericsson-4471322400
Published data: 2026-09-28 04:00:00+00:00

*********************** 2 **************************
Title: Ericsson söker Master Thesis: Resource Orchestration for Agentic AI Systems i Stockholm | LinkedIn
URL: https://www.linkedin.com/jobs/view/master-thesis-resource-orchestration-for-agentic-ai-systems-at-ericsson-4471305538
Published data: 2026-09-27 22:00:00+00:00

*********************** 3 **************************
Title: Master Thesis: Brain-Inspired Algorithms for Radio Access Networks på Ericsson – Stockholm | LinkedIn Jobb
URL: https://www.linkedin.com/jobs/view/master-thesis-brain-inspired-algorithms-for-radio-access-networks-at-ericsson-4471304514
Published d

In [7]:
unseen_jobs = filter_unseen_jobs(res, should_print=True)

Existing Job IDs: 
b48ef59f2c59546f, bd60c7b6d493ccc8, ac6b3874f3b39880, f7ff516b53dec0e5, ef5d5c0c56916c38
New Job IDs: 
e2d76fb307a11636


In [8]:
unseen_jobs

JobSearchResponse(jobs=[JobSearch(external_id='e2d76fb307a11636', title='Master Thesis: Quantum Computing Algorithms for Telecom', url='https://www.linkedin.com/jobs/view/master-thesis-quantum-computing-algorithms-for-telecom-at-ericsson-4471303533', content='This master thesis project aims at telco-specific optimization using quantum computing algorithms. Primary country and city: Sweden (SE)', raw_content='', published_datetime=datetime.datetime(2026, 9, 27, 22, 0, tzinfo=datetime.timezone.utc))], queries=['se.linkedin.com/jobs/ "Master Thesis AI" sweden'], total_results=6)

In [9]:
res_with_content = scrape_raw_job_content(unseen_jobs)

In [10]:
jobs = extract_jobs_info(res_with_content, print_jobs=True)


External id: e2d76fb307a11636
Title: Master Thesis: Quantum Computing Algorithms for Telecom
Company: Ericsson
Posted date: 2026-09-27 22:00:00+00:00
Summary: An onsite thesis role for students in Stockholm, Sweden. This quantum computing, machine learning, and telecommunications position sits within the telecommunications industry, where the team explores quantum-based solutions for large-scale network management. The student will develop and evaluate quantum computing algorithms to optimize radio access network workloads, comparing these new approaches against conventional methods. Candidates need a Master's degree in Computer Science, Engineering Physics, Machine Learning, or Mathematics, along with proficiency in Probability Theory, Neural Networks, System Modeling, and programming in C++ and Python. The work is expected to start in January 2027 for a duration of six months.

Embedding Text: Title: Master Thesis: Quantum Computing Algorithms for Telecom
Company: Ericsson
Specializ

In [11]:
saved_jobs_ids = Container.job_repository.save_jobs(jobs=jobs)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
e:\MS in CS\Semester-3\Period 1\LLM\group_project\.venv\Lib\site-packages\transformers\modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


In [12]:
saved_jobs_ids

['e2d76fb307a11636']